# 01 — Dataset Conversion: YOLO Pose → COCO JSON

**Tujuan:** Konversi dataset `soccer-field-localization` dari format YOLO Pose
(.txt labels) ke format COCO JSON yang dibutuhkan oleh MMPose (HRNet, ViTPose, DETR).

**Runtime:** CPU-only. Tidak butuh GPU. Estimasi waktu: ~5-10 menit.

**Output:**
- `data/converted/annotations/train.json`
- `data/converted/annotations/val.json`
- `data/converted/annotations/test.json`

**Catatan:** Jalankan notebook ini SEKALI sebelum training HRNet/ViTPose/DETR.
Setelah selesai, verifikasi dengan bagian `pycocotools` di bawah.

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install pycocotools pyyaml tqdm opencv-python-headless --quiet

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Environment & Path Setup — kompatibel Kaggle DAN Colab (auto-detect)
# ══════════════════════════════════════════════════════════════════════
import os, sys

def _detect_env():
    if os.path.exists('/kaggle/working'):
        return 'kaggle'
    if os.path.exists('/content'):
        return 'colab'
    return 'local'

ENV = _detect_env()
print(f'Environment terdeteksi: {ENV}')

# ⚠️ GANTI dengan URL repo GitHub kamu sendiri (isi SEKALI saja per notebook)
GITHUB_REPO = 'https://github.com/atangorp/soccer-homography-benchmark.git'

def _link(src, dst):
    """Symlink src -> dst kalau src ada dan dst belum ada. Aman dipanggil berkali-kali."""
    if os.path.exists(src) and not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        os.symlink(src, dst)

if ENV == 'kaggle':
    PROJECT_ROOT = '/kaggle/working/soccer-homography-benchmark'
    if not os.path.exists(PROJECT_ROOT):
        os.system(f'git clone -q {GITHUB_REPO} "{PROJECT_ROOT}"')
    sys.path.insert(0, PROJECT_ROOT)

    # WAJIB: attach dataset "soccer-field-raw" via "+ Add Input" di sidebar kanan
    # (ganti 'soccer-field-raw' kalau slug dataset kamu berbeda)
    _link('/kaggle/input/soccer-field-raw',
          f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8')

elif ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/soccer-homography-benchmark'
    sys.path.insert(0, PROJECT_ROOT)

else:
    PROJECT_ROOT = os.getcwd()
    sys.path.insert(0, PROJECT_ROOT)

YOLO_DIR   = f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8'
OUTPUT_DIR = f'{PROJECT_ROOT}/data/converted/annotations'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'PROJECT_ROOT : {PROJECT_ROOT}')


In [ ]:
# ── Cell 4: Import converter ─────────────────────────────────────────────────
from src.data.yolo_to_coco import convert_all_splits, verify_coco_json, compare_stats

print("✅ Import berhasil")

In [ ]:
# ── Cell 5: Konversi semua split ─────────────────────────────────────────────
# Ini mungkin memakan waktu 5-10 menit tergantung ukuran dataset.
# Progress bar akan muncul untuk setiap split.

output_paths = convert_all_splits(
    yolo_dir=YOLO_DIR,
    output_dir=OUTPUT_DIR,
    splits=["train", "val", "test"],
    n_keypoints=32,
)

print("\n🎉 Konversi selesai!")
for split, path in output_paths.items():
    size_mb = os.path.getsize(path) / 1024**2
    print(f"   {split:5s}: {path} ({size_mb:.1f} MB)")

In [ ]:
# ── Cell 6: VERIFIKASI dengan pycocotools ────────────────────────────────────
# WAJIB dijalankan sebelum training HRNet/ViTPose/DETR.
# Jika ada error di sini, JANGAN lanjutkan training.

all_valid = True
for split, path in output_paths.items():
    valid = verify_coco_json(path, verbose=True)
    if not valid:
        all_valid = False
        print(f"❌ {split}.json GAGAL verifikasi!")

if all_valid:
    print("\n✅ Semua file COCO JSON valid. Siap untuk training!")
else:
    print("\n❌ Ada file yang invalid. Cek error di atas dan jalankan ulang Cell 5.")

In [ ]:
# ── Cell 7: Perbandingan statistik YOLO vs COCO ──────────────────────────────
# Verifikasi tidak ada data yang hilang atau bergeser saat konversi.
# Semua baris harus menunjukkan ✅

for split in ["train", "val", "test"]:
    coco_path = os.path.join(OUTPUT_DIR, f"{split}.json")
    if os.path.exists(coco_path):
        compare_stats(YOLO_DIR, coco_path, split=split)

In [ ]:
# ── Cell 8: Simpan metadata konversi ────────────────────────────────────────
import json
from datetime import datetime

meta = {
    "converted_at": datetime.now().isoformat(),
    "source_format": "YOLO Pose (.txt + data.yaml)",
    "target_format": "COCO JSON",
    "n_keypoints": 32,
    "splits_converted": list(output_paths.keys()),
    "output_files": output_paths,
    "verified": all_valid,
}

meta_path = os.path.join(OUTPUT_DIR, "conversion_meta.json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"✅ Metadata tersimpan: {meta_path}")
print("\n📌 Langkah selanjutnya:")
print("   → Buka notebooks/02b_train_hrnet.ipynb")
print("   → Set COCO_ANN_DIR ke path ini:", OUTPUT_DIR)